<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/06_efficientnet_b0_final_development_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Stage 6 — Final EfficientNet-B0 Development Training

## Objective

Train a single final EfficientNet-B0 binary breast histopathology classifier using all 49 patients in the frozen BreaKHis Development partition.

The architecture and training protocol were established during Stage 5 through three-fold patient-grouped Development experiments. EfficientNet-B0 was selected based on its overall discrimination and probability-quality results, while acknowledging uncertainty in the architecture comparison.

## Locked Training Protocol

- **Architecture:** ImageNet-pretrained EfficientNet-B0 with a one-logit binary classifier.
- **Trainable components:** `features[7]`, `features[8]`, and the final classifier layer.
- **Training data:** All 49 Development patients, comprising 1,133 images and 17,895 patches at 40× magnification.
- **Training duration:** Exactly five epochs, informed by Development checkpoint selections of epochs 5, 5, and 4.
- **Optimization:** AdamW, class-weighted binary cross-entropy, and the previously established learning rates and regularization.
- **Preprocessing:** The same tissue-patch extraction, augmentation, and normalization procedures used in the formal Development experiments.
- **Reproducibility:** Fixed random seed 631 and documented environment and model artifacts.

The final checkpoint will be the model state after epoch 5. No new validation-based epoch selection or hyperparameter tuning will occur.

## Data Isolation

Only the frozen Development partition may be used for model fitting.

The separate Calibration cohort will later be used exclusively for fitting post-hoc calibration methods. The independent Test cohort and external datasets will remain untouched until their designated evaluation stages.

## Outputs

This stage will produce the final model checkpoint, training history, configuration, dataset-integrity verification, and checkpoint reproducibility records. The model weights will be stored separately from the GitHub repository.


In [1]:

# ============================================================
# Environment Setup and Frozen Development Audit
# ============================================================

import json
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Reproducibility settings
# ------------------------------------------------------------

SEED = 631

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), (
    "GPU unavailable. In Colab, select "
    "Runtime > Change runtime type > GPU."
)

DEVICE = torch.device("cuda")

print("=" * 70)
print("STAGE 6 — ENVIRONMENT CHECK")
print("=" * 70)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Random seed:", SEED)

# ------------------------------------------------------------
# 2. Locate or clone the research repository
# ------------------------------------------------------------

REPO_URL = (
    "https://github.com/Yisihaq01/"
    "breast-histology-calibration.git"
)

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

if not REPO_ROOT.exists():

    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            REPO_URL,
            str(REPO_ROOT)
        ],
        check=True
    )

    print("\nRepository cloned successfully.")

else:

    print("\nExisting repository found.")
    print("No files were overwritten or updated.")

assert (REPO_ROOT / ".git").is_dir()

commit_hash = subprocess.check_output(
    [
        "git", "-C", str(REPO_ROOT),
        "rev-parse", "--short", "HEAD"
    ],
    text=True
).strip()

print("Repository:", REPO_ROOT)
print("Git commit:", commit_hash)

# ------------------------------------------------------------
# 3. Verify completed Stage 5 architecture decision
# ------------------------------------------------------------

decision_path = (
    REPO_ROOT /
    "results/formal_development/efficientnet_b0/"
    "cross_fold_analysis/stage5_architecture_selection.json"
)

assert decision_path.is_file(), (
    "Stage 5 decision record missing. "
    "Ensure the completed Stage 5 results "
    "are committed and pushed to GitHub."
)

with open(
    decision_path,
    "r",
    encoding="utf-8"
) as f:
    stage5_decision = json.load(f)

assert (
    stage5_decision["selected_primary_architecture"]
    == "efficientnet_b0"
)

assert stage5_decision["development_patients"] == 49
assert stage5_decision["development_images"] == 1133
assert stage5_decision["development_patches"] == 17895

assert (
    stage5_decision["stage6_plan"]["epochs"] == 5
)

assert (
    stage5_decision["stage6_plan"]["architecture"]
    == "efficientnet_b0"
)

print("\nPASS: Stage 5 architecture decision verified.")
print("Selected architecture: EfficientNet-B0")
print("Locked training epochs: 5")

# ------------------------------------------------------------
# 4. Load only the frozen Development patch manifest
# ------------------------------------------------------------

manifest_path = (
    REPO_ROOT /
    "Metadata/breakhis_development_all_patch_manifest.csv"
)

assert manifest_path.is_file(), (
    f"Development manifest missing: {manifest_path}"
)

manifest = pd.read_csv(
    manifest_path,
    dtype={"patient_id": str}
)

required_columns = {
    "filename",
    "relative_path",
    "patient_id",
    "label",
    "development_fold",
    "patch_index"
}

assert required_columns.issubset(
    manifest.columns
), (
    "Missing manifest columns: "
    f"{required_columns - set(manifest.columns)}"
)

manifest["development_fold"] = (
    manifest["development_fold"].astype(int)
)

# ------------------------------------------------------------
# 5. Verify Development membership and uniqueness
# ------------------------------------------------------------

assert len(manifest) == 17895

assert manifest["patient_id"].nunique() == 49

assert manifest["filename"].nunique() == 1133

assert set(
    manifest["development_fold"].unique()
) == {1, 2, 3}

assert set(manifest["label"].unique()) == {0, 1}

assert not manifest[
    ["filename", "patch_index"]
].duplicated().any()

assert (
    manifest.groupby("patient_id")[
        "development_fold"
    ].nunique() == 1
).all()

assert (
    manifest.groupby("filename")[
        "patient_id"
    ].nunique() == 1
).all()

# ------------------------------------------------------------
# 6. Independently verify original fold composition
# ------------------------------------------------------------

fold_audit = (
    manifest.groupby("development_fold")
    .agg(
        patients=("patient_id", "nunique"),
        images=("filename", "nunique"),
        patches=("patch_index", "size")
    )
    .reset_index()
)

expected = {
    1: (17, 372, 5861),
    2: (16, 431, 6810),
    3: (16, 330, 5224)
}

for row in fold_audit.itertuples(index=False):

    actual = (
        int(row.patients),
        int(row.images),
        int(row.patches)
    )

    assert actual == expected[int(row.development_fold)]

print("\n" + "=" * 70)
print("FROZEN DEVELOPMENT DATASET")
print("=" * 70)

print(fold_audit.to_string(index=False))

print("\nTotal patients:", manifest["patient_id"].nunique())
print("Total images:", manifest["filename"].nunique())
print("Total patches:", len(manifest))

# ------------------------------------------------------------
# 7. Final preflight checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 6 — INITIAL PREFLIGHT")
print("=" * 70)

print("PASS: GPU is available.")
print("PASS: Random seed initialized.")
print("PASS: Repository located.")
print("PASS: Stage 5 decision loaded.")
print("PASS: EfficientNet-B0 architecture confirmed.")
print("PASS: Five-epoch training budget confirmed.")
print("PASS: All 49 Development patients verified.")
print("PASS: All 1,133 source images verified.")
print("PASS: All 17,895 patches verified.")
print("PASS: Original patient-grouped folds preserved.")
print("PASS: Calibration and Test datasets not accessed.")
print("PASS: Ready for Stage 6 data preparation.")


STAGE 6 — ENVIRONMENT CHECK
Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Random seed: 631

Repository cloned successfully.
Repository: /content/breast-histology-calibration
Git commit: f479f7f

PASS: Stage 5 architecture decision verified.
Selected architecture: EfficientNet-B0
Locked training epochs: 5

FROZEN DEVELOPMENT DATASET
 development_fold  patients  images  patches
                1        17     372     5861
                2        16     431     6810
                3        16     330     5224

Total patients: 49
Total images: 1133
Total patches: 17895

STAGE 6 — INITIAL PREFLIGHT
PASS: GPU is available.
PASS: Random seed initialized.
PASS: Repository located.
PASS: Stage 5 decision loaded.
PASS: EfficientNet-B0 architecture confirmed.
PASS: Five-epoch training budget confirmed.
PASS: All 49 Development patients verified.
PASS: All 1,133 source images verified.
PASS: All 17,895 patches verified.
PASS: Original patient-grouped folds preserved.
P

In [2]:

# ============================================================
# BreaKHis Dataset Restoration and Training-Data Audit
# ============================================================

import sys
import subprocess
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Make KaggleHub available
# ------------------------------------------------------------

try:
    import kagglehub
except ImportError:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "-q", "kagglehub"
        ],
        check=True
    )
    kagglehub = importlib.import_module("kagglehub")

# ------------------------------------------------------------
# 2. Restore the BreaKHis source dataset
# ------------------------------------------------------------

downloaded_root = Path(
    kagglehub.dataset_download("ambarish/breakhis")
)

assert downloaded_root.is_dir()

print("=" * 70)
print("STAGE 6 — BREAKHIS DATASET")
print("=" * 70)

print("KaggleHub dataset location:", downloaded_root)

# ------------------------------------------------------------
# 3. Identify the path prefix used by our frozen manifest
# ------------------------------------------------------------

development_images = (
    manifest[
        ["filename", "relative_path", "patient_id", "label"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

assert len(development_images) == 1133
assert development_images["filename"].is_unique
assert development_images["relative_path"].notna().all()

relative_paths = [
    Path(str(value))
    for value in development_images["relative_path"]
]

# Manifest paths must remain relative to a dataset root.
assert all(
    not path.is_absolute() and ".." not in path.parts
    for path in relative_paths
)

sample_relative_path = relative_paths[0]

candidate_roots = [
    downloaded_root,
    downloaded_root / "BreaKHis_v1",
    downloaded_root / "BreaKHis",
    downloaded_root / "breakhis",
    downloaded_root / "BreaKHis_v1" / "BreaKHis_v1"
]

matching_roots = [
    root
    for root in candidate_roots
    if (root / sample_relative_path).is_file()
]

assert matching_roots, (
    "Could not locate the first Development image. "
    f"Manifest path: {sample_relative_path}. "
    f"Download root: {downloaded_root}"
)

DATASET_ROOT = matching_roots[0]

print("Resolved image root:", DATASET_ROOT)
print("Sample relative path:", sample_relative_path)

# ------------------------------------------------------------
# 4. Verify every frozen Development source image
# ------------------------------------------------------------

development_images["source_path"] = [
    str(DATASET_ROOT / relative_path)
    for relative_path in relative_paths
]

missing_paths = [
    path
    for path in development_images["source_path"]
    if not Path(path).is_file()
]

assert len(missing_paths) == 0, (
    f"{len(missing_paths)} Development images are missing. "
    f"First missing path: {missing_paths[:1]}"
)

assert development_images["patient_id"].nunique() == 49

assert len(development_images) == (
    manifest["filename"].nunique()
)

print("\nPASS: All 1,133 Development image files located.")
print("PASS: Frozen filename-to-path mapping preserved.")

# ------------------------------------------------------------
# 5. Verify labels and calculate full-Development class weight
# ------------------------------------------------------------

assert set(manifest["label"].unique()) == {0, 1}

class_audit = (
    manifest.groupby("label")
    .agg(
        patients=("patient_id", "nunique"),
        images=("filename", "nunique"),
        patches=("patch_index", "size")
    )
    .reset_index()
)

benign_patches = int(
    class_audit.loc[
        class_audit["label"] == 0,
        "patches"
    ].iloc[0]
)

malignant_patches = int(
    class_audit.loc[
        class_audit["label"] == 1,
        "patches"
    ].iloc[0]
)

assert benign_patches > 0
assert malignant_patches > 0

assert benign_patches + malignant_patches == 17895

# Same training-only positive-class weighting rule
# as in Stage 5, recalculated for the entire Development set.
FINAL_POS_WEIGHT = (
    benign_patches / malignant_patches
)

assert np.isfinite(FINAL_POS_WEIGHT)
assert FINAL_POS_WEIGHT > 0

print("\n" + "=" * 70)
print("FULL-DEVELOPMENT CLASS DISTRIBUTION")
print("=" * 70)

print(class_audit.to_string(index=False))

print(
    "\nFinal malignant-class BCE pos_weight:",
    round(FINAL_POS_WEIGHT, 6)
)

# ------------------------------------------------------------
# 6. Recheck patient isolation and patch membership
# ------------------------------------------------------------

assert len(manifest) == 17895

assert not manifest[
    ["filename", "patch_index"]
].duplicated().any()

assert (
    manifest.groupby("filename")["label"].nunique()
    == 1
).all()

assert (
    manifest.groupby("filename")["patient_id"].nunique()
    == 1
).all()

assert (
    manifest.groupby("patient_id")[
        "development_fold"
    ].nunique() == 1
).all()

print("\n" + "=" * 70)
print("STAGE 6 — DATA RESTORATION AUDIT")
print("=" * 70)

print("PASS: Original BreaKHis source restored.")
print("PASS: All 49 Development patients represented.")
print("PASS: All 1,133 Development image paths verified.")
print("PASS: All 17,895 patch records verified.")
print("PASS: Binary label consistency verified.")
print("PASS: Full-Development class weight calculated.")
print("PASS: No duplicate image-patch identities.")
print("PASS: Frozen patient assignments preserved.")
print("PASS: Calibration and Test data not used for training.")

print("\nReady to restore Stage 5 preprocessing functions.")


Using Colab cache for faster access to the 'breakhis' dataset.
STAGE 6 — BREAKHIS DATASET
KaggleHub dataset location: /kaggle/input/breakhis
Resolved image root: /kaggle/input/breakhis
Sample relative path: BreaKHis_v1/BreaKHis_v1/histology_slides/breast/benign/SOB/tubular_adenoma/SOB_B_TA_14-13200/40X/SOB_B_TA-14-13200-40-001.png

PASS: All 1,133 Development image files located.
PASS: Frozen filename-to-path mapping preserved.

FULL-DEVELOPMENT CLASS DISTRIBUTION
 label  patients  images  patches
     0        14     329     5178
     1        35     804    12717

Final malignant-class BCE pos_weight: 0.407172

STAGE 6 — DATA RESTORATION AUDIT
PASS: Original BreaKHis source restored.
PASS: All 49 Development patients represented.
PASS: All 1,133 Development image paths verified.
PASS: All 17,895 patch records verified.
PASS: Binary label consistency verified.
PASS: Full-Development class weight calculated.
PASS: No duplicate image-patch identities.
PASS: Frozen patient assignments pre

In [7]:

# ============================================================
# Restore Original Preprocessing and Source Dependencies
# ============================================================

import ast
import hashlib
import inspect
import json
import math
import random
import traceback

from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

import numpy as np
import pandas as pd
import torch

from PIL import Image, ImageOps, ImageFilter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import functional as TF

# ------------------------------------------------------------
# 1. Load the original notebook
# ------------------------------------------------------------

SOURCE_NOTEBOOK = (
    REPO_ROOT /
    "Notebooks/03_breakhis_formal_development.ipynb"
)

assert SOURCE_NOTEBOOK.is_file()

with open(SOURCE_NOTEBOOK, "r", encoding="utf-8") as f:
    notebook = json.load(f)

required_names = [
    "resample_to_target_mpp",
    "RandomRightAngleRotation",
    "BreakHisPatchDataset"
]

# ------------------------------------------------------------
# 2. Index source definitions, imports and assignments
# ------------------------------------------------------------

source_index = {}
source_locations = {}
source_definitions = {}

code_cell_count = 0

for cell in notebook["cells"]:

    if cell.get("cell_type") != "code":
        continue

    code_cell_count += 1

    source = cell.get("source", "")

    if isinstance(source, list):
        source = "".join(source)

    try:
        tree = ast.parse(source)
    except SyntaxError:
        # Notebook magics are not executed.
        continue

    for node in tree.body:

        bound_names = []

        if isinstance(
            node, (ast.FunctionDef, ast.ClassDef)
        ):
            bound_names = [node.name]
            source_definitions[node.name] = node

        elif isinstance(node, ast.Import):
            bound_names = [
                alias.asname or alias.name.split(".")[0]
                for alias in node.names
            ]

        elif isinstance(node, ast.ImportFrom):
            if node.module == "__future__":
                continue

            bound_names = [
                alias.asname or alias.name
                for alias in node.names
                if alias.name != "*"
            ]

        elif isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name):
                    bound_names.append(target.id)

        elif isinstance(node, ast.AnnAssign):
            if isinstance(node.target, ast.Name):
                bound_names = [node.target.id]

        for name in bound_names:
            source_index[name] = node
            source_locations[name] = code_cell_count

for name in required_names:
    assert name in source_definitions, (
        f"Missing source definition: {name}"
    )

print("=" * 70)
print("STAGE 6 — ORIGINAL PREPROCESSING RESTORATION")
print("=" * 70)

print("Source notebook:", SOURCE_NOTEBOOK.name)
print("Original code cells:", code_cell_count)
print("PASS: All required source definitions found.")

# ------------------------------------------------------------
# 3. Restore dependencies recursively
# ------------------------------------------------------------

restored_names = set()
resolution_stack = set()

def restore_source_name(name, force=False):

    if name in restored_names:
        return

    # Existing standard imports need not be replaced.
    if not force and name in globals():
        return

    if name in resolution_stack:
        raise RuntimeError(
            f"Circular source dependency involving {name}"
        )

    if name not in source_index:
        raise NameError(
            f"'{name}' was not found among source "
            "notebook definitions, imports or assignments."
        )

    node = source_index[name]

    # Never run arbitrary notebook cells.
    # Only the selected AST statement is executed.
    if isinstance(node, (ast.Assign, ast.AnnAssign)):

        expression = node.value

        if expression is None:
            raise RuntimeError(
                f"Source assignment for {name} has no value."
            )

        # Reject function calls in assignments so loading
        # a constant cannot inadvertently trigger training
        # or dataset processing.
        if any(
            isinstance(child, ast.Call)
            for child in ast.walk(expression)
        ):
            raise RuntimeError(
                f"Source assignment for {name} contains "
                "a function call and requires inspection."
            )

    resolution_stack.add(name)

    try:

        for attempt in range(30):

            module = ast.Module(
                body=[node],
                type_ignores=[]
            )

            ast.fix_missing_locations(module)

            try:
                exec(
                    compile(
                        module,
                        str(SOURCE_NOTEBOOK),
                        "exec"
                    ),
                    globals(),
                    globals()
                )

                break

            except NameError as error:

                dependency = error.name

                if dependency is None:
                    raise

                if dependency == name:
                    raise RuntimeError(
                        f"Self-referential definition: {name}"
                    ) from error

                restore_source_name(dependency)

        else:
            raise RuntimeError(
                f"Dependency resolution limit reached: {name}"
            )

        assert name in globals(), (
            f"Definition did not create {name}"
        )

        restored_names.add(name)

    finally:
        resolution_stack.remove(name)


# ------------------------------------------------------------
# 4. Restore microscopy configuration first
# ------------------------------------------------------------

# These defaults must be obtained from the original
# notebook, not assigned guessed values.

for constant in [
    "SOURCE_MPP_BREAKHIS",
    "TARGET_MPP"
]:
    restore_source_name(
        constant,
        force=True
    )

    value = globals()[constant]

    assert isinstance(
        value, (int, float, np.number)
    )

    assert np.isfinite(value)
    assert value > 0

    print(
        f"RESTORED CONSTANT: {constant} = {value}"
    )

# Target resolution was documented as 0.50 um/pixel.
assert np.isclose(
    float(TARGET_MPP),
    0.50,
    atol=1e-12
), "Original target resolution differs from expected 0.50."

print("\nPASS: Microscopy constants restored from source.")

# ------------------------------------------------------------
# 5. Restore the exact original definitions
# ------------------------------------------------------------

for name in required_names:

    restore_source_name(
        name,
        force=True
    )

    node = source_definitions[name]

    source_text = ast.unparse(node)

    sha256 = hashlib.sha256(
        source_text.encode("utf-8")
    ).hexdigest()

    print(f"\nRESTORED: {name}")
    print("Original cell:", source_locations[name])
    print("Source SHA-256:", sha256)

# ------------------------------------------------------------
# 6. Resolve global dependencies used by class methods
# ------------------------------------------------------------

# A class may restore successfully even when a method
# references a global constant or helper that will only
# be needed later when the method is called.
#
# Check those references now to prevent another error
# during DataLoader construction or patch extraction.

def source_functions_to_inspect():

    functions = [resample_to_target_mpp]

    for cls in [
        RandomRightAngleRotation,
        BreakHisPatchDataset
    ]:
        for attribute in cls.__dict__.values():

            if inspect.isfunction(attribute):
                functions.append(attribute)

    return functions


for pass_index in range(10):

    newly_restored = 0

    for function in source_functions_to_inspect():

        closure = inspect.getclosurevars(
            function
        )

        for name in sorted(closure.unbound):

            # Some attribute names can appear as
            # unbound in code introspection.
            # Resolve only actual notebook definitions.
            if (
                name not in globals()
                and name in source_index
            ):
                restore_source_name(name)
                newly_restored += 1

    if newly_restored == 0:
        break

print("\nPASS: Available source-defined method dependencies restored.")

# ------------------------------------------------------------
# 7. Verify types and signatures
# ------------------------------------------------------------

assert inspect.isfunction(
    resample_to_target_mpp
)

assert inspect.isclass(
    RandomRightAngleRotation
)

assert inspect.isclass(
    BreakHisPatchDataset
)

assert issubclass(
    BreakHisPatchDataset,
    Dataset
)

signature = inspect.signature(
    resample_to_target_mpp
)

assert np.isclose(
    signature.parameters["source_mpp"].default,
    SOURCE_MPP_BREAKHIS
)

assert np.isclose(
    signature.parameters["target_mpp"].default,
    TARGET_MPP
)

print("\n" + "=" * 70)
print("VERIFIED ORIGINAL SIGNATURES")
print("=" * 70)

for name in required_names:

    print(
        f"\n{name}:",
        inspect.signature(globals()[name])
    )

# ------------------------------------------------------------
# 8. Preserve Stage 6 data isolation
# ------------------------------------------------------------

assert len(manifest) == 17895
assert manifest["filename"].nunique() == 1133
assert manifest["patient_id"].nunique() == 49

assert abs(
    FINAL_POS_WEIGHT - (5178 / 12717)
) < 1e-10

assert DATASET_ROOT.is_dir()

# ------------------------------------------------------------
# 9. Final report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 6 — PREPROCESSING RESTORATION AUDIT")
print("=" * 70)

print("PASS: Original microscopy constants restored.")
print("PASS: Original resampling function restored.")
print("PASS: Original rotation transform restored.")
print("PASS: Original patch dataset restored.")
print("PASS: Source-defined dependencies checked.")
print("PASS: Original function signatures verified.")
print("PASS: Frozen Development manifest unchanged.")
print("PASS: Full-Development class weight unchanged.")
print("PASS: No Calibration or Test data accessed.")
print("PASS: No model training performed.")

print("\nReady for full-Development dataset construction.")


STAGE 6 — ORIGINAL PREPROCESSING RESTORATION
Source notebook: 03_breakhis_formal_development.ipynb
Original code cells: 52
PASS: All required source definitions found.
RESTORED CONSTANT: SOURCE_MPP_BREAKHIS = 0.49
RESTORED CONSTANT: TARGET_MPP = 0.5

PASS: Microscopy constants restored from source.

RESTORED: resample_to_target_mpp
Original cell: 17
Source SHA-256: eb3b136fc7c45e45d62c710c99c55a8849eac3da7711eb4fdb2b5e2b2d79fd73

RESTORED: RandomRightAngleRotation
Original cell: 17
Source SHA-256: c11e1adfd904ab858ef93e667cd6c3fc5319a9c5024d222f0aa29ed9eb7f6d8c

RESTORED: BreakHisPatchDataset
Original cell: 18
Source SHA-256: 4b74bf1e15c939fbcc74b042401767f942a0abdfc8141abf1445da5da5112a22

PASS: Available source-defined method dependencies restored.

VERIFIED ORIGINAL SIGNATURES

resample_to_target_mpp: (image, source_mpp=0.49, target_mpp=0.5)

RandomRightAngleRotation: ()

BreakHisPatchDataset: (manifest, dataset_root, transform=None)

STAGE 6 — PREPROCESSING RESTORATION AUDIT
PASS: 

In [8]:

# ============================================================
# Final Development Dataset and DataLoader Construction
# ============================================================

import math
import random

import numpy as np
import torch

from torch.utils.data import DataLoader
from torchvision import transforms
from IPython.display import display

# ------------------------------------------------------------
# 1. Final training configuration
# ------------------------------------------------------------

FINAL_BATCH_SIZE = 32
FINAL_EPOCHS = 5
FINAL_NUM_WORKERS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

assert SEED == 631
assert FINAL_EPOCHS == 5
assert len(manifest) == 17895

# ------------------------------------------------------------
# 2. Reconstruct established training augmentation
# ------------------------------------------------------------

final_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    RandomRightAngleRotation(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print("=" * 70)
print("STAGE 6 — FINAL TRAINING AUGMENTATION")
print("=" * 70)

print(final_train_transform)

# ------------------------------------------------------------
# 3. Construct full Development training dataset
# ------------------------------------------------------------

final_train_manifest = (
    manifest.copy()
    .reset_index(drop=True)
)

assert len(final_train_manifest) == 17895

assert (
    final_train_manifest["patient_id"].nunique()
    == 49
)

assert (
    final_train_manifest["filename"].nunique()
    == 1133
)

final_train_dataset = BreakHisPatchDataset(
    manifest=final_train_manifest,
    dataset_root=DATASET_ROOT,
    transform=final_train_transform
)

assert len(final_train_dataset) == 17895

print("\n" + "=" * 70)
print("FULL DEVELOPMENT TRAINING DATASET")
print("=" * 70)

print("Patients:", final_train_manifest["patient_id"].nunique())
print("Images:", final_train_manifest["filename"].nunique())
print("Patches:", len(final_train_dataset))
print("Batch size:", FINAL_BATCH_SIZE)
print("Training epochs:", FINAL_EPOCHS)
print("Positive-class weight:", round(FINAL_POS_WEIGHT, 6))

# ------------------------------------------------------------
# 4. Inspect one patch from the restored dataset
# ------------------------------------------------------------

sample = final_train_dataset[0]

assert isinstance(sample, (tuple, list, dict)), (
    "Unexpected dataset output structure."
)

if isinstance(sample, dict):

    sample_image = sample.get("image")

    if sample_image is None:
        sample_image = sample.get("pixel_values")

    sample_label = sample.get("label")

else:

    assert len(sample) >= 2

    sample_image = sample[0]
    sample_label = sample[1]

assert isinstance(sample_image, torch.Tensor)

assert sample_image.shape == (3, 224, 224), (
    f"Unexpected image shape: {sample_image.shape}"
)

assert torch.isfinite(sample_image).all()

sample_label_value = float(
    torch.as_tensor(sample_label).item()
)

assert sample_label_value in (0.0, 1.0)

print("\nPASS: Original patch dataset produced valid tensors.")
print("Sample image shape:", tuple(sample_image.shape))
print("Sample label:", sample_label_value)
print("Sample tensor dtype:", sample_image.dtype)

# ------------------------------------------------------------
# 5. Set reproducible DataLoader worker seeds
# ------------------------------------------------------------

def seed_final_training_worker(worker_id):

    worker_seed = (
        torch.initial_seed() % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


final_train_generator = torch.Generator()
final_train_generator.manual_seed(SEED)

# ------------------------------------------------------------
# 6. Construct the final training DataLoader
# ------------------------------------------------------------

final_train_loader = DataLoader(
    dataset=final_train_dataset,
    batch_size=FINAL_BATCH_SIZE,
    shuffle=True,
    num_workers=FINAL_NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=FINAL_NUM_WORKERS > 0,
    worker_init_fn=seed_final_training_worker,
    generator=final_train_generator
)

expected_batches = math.ceil(
    17895 / FINAL_BATCH_SIZE
)

assert len(final_train_loader) == expected_batches

print("\nTraining batches per epoch:", len(final_train_loader))
print("Expected batches:", expected_batches)

# ------------------------------------------------------------
# 7. Verify one actual training batch
# ------------------------------------------------------------

batch = next(iter(final_train_loader))

if isinstance(batch, dict):

    batch_images = batch.get("image")

    if batch_images is None:
        batch_images = batch.get("pixel_values")

    batch_labels = batch.get("label")

else:

    assert isinstance(batch, (tuple, list))
    assert len(batch) >= 2

    batch_images = batch[0]
    batch_labels = batch[1]

assert isinstance(batch_images, torch.Tensor)

assert batch_images.shape == (
    FINAL_BATCH_SIZE, 3, 224, 224
)

assert torch.isfinite(batch_images).all()

batch_labels_tensor = torch.as_tensor(
    batch_labels
).reshape(-1)

assert batch_labels_tensor.shape[0] == FINAL_BATCH_SIZE

assert torch.isin(
    batch_labels_tensor,
    torch.tensor(
        [0, 1],
        dtype=batch_labels_tensor.dtype
    )
).all()

print("\n" + "=" * 70)
print("TRAINING BATCH VERIFICATION")
print("=" * 70)

print("Image batch shape:", tuple(batch_images.shape))
print("Label batch shape:", tuple(batch_labels_tensor.shape))
print("Image dtype:", batch_images.dtype)
print("Label dtype:", batch_labels_tensor.dtype)
print("Finite values:", bool(torch.isfinite(batch_images).all()))

# ------------------------------------------------------------
# 8. Final audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 6 — TRAINING DATASET AUDIT")
print("=" * 70)

print("PASS: Full Development manifest loaded.")
print("PASS: All 49 Development patients included.")
print("PASS: All 1,133 Development images included.")
print("PASS: All 17,895 patches included.")
print("PASS: Training augmentations constructed.")
print("PASS: Original patch dataset reused.")
print("PASS: Training DataLoader constructed.")
print("PASS: Sample batch shape verified.")
print("PASS: Binary labels verified.")
print("PASS: Random generator initialized.")
print("PASS: Calibration and Test data not accessed.")
print("PASS: Ready for final EfficientNet-B0 initialization.")


STAGE 6 — FINAL TRAINING AUGMENTATION
Compose(
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

FULL DEVELOPMENT TRAINING DATASET
Patients: 49
Images: 1133
Patches: 17895
Batch size: 32
Training epochs: 5
Positive-class weight: 0.407172

PASS: Original patch dataset produced valid tensors.
Sample image shape: (3, 224, 224)
Sample label: 0.0
Sample tensor dtype: torch.float32

Training batches per epoch: 560
Expected batches: 560

TRAINING BATCH VERIFICATION
Image batch shape: (32, 3, 224, 224)
Label batch shape: (32,)
Image dtype: torch.float32
Label dtype: torch.float32
Finite values: True

STAGE 6 — TRAINING DATASET AUDIT
PASS: Full Development manifest loaded.
PASS: All 49 Development patients included.
PASS: All 1,133 Development images included.
PASS: All 17,895 patches included.
PASS: Training augmentations constructed.
PASS: Original patch dataset reused.
PASS: Training DataLoader

In [9]:

# ============================================================
# Final EfficientNet-B0 Initialization and Configuration
# ============================================================

import random
import numpy as np
import torch
import torch.nn as nn

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

# ------------------------------------------------------------
# 1. Locked Stage 6 hyperparameters
# ------------------------------------------------------------

FINAL_SEED = 631
FINAL_FEATURE_LR = 1e-4
FINAL_CLASSIFIER_LR = 5e-4
FINAL_WEIGHT_DECAY = 1e-4

assert FINAL_EPOCHS == 5
assert FINAL_BATCH_SIZE == 32
assert FINAL_SEED == SEED
assert len(final_train_dataset) == 17895

# Reset RNG after sample-batch inspection.
random.seed(FINAL_SEED)
np.random.seed(FINAL_SEED)
torch.manual_seed(FINAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(FINAL_SEED)

print("=" * 70)
print("STAGE 6 — FINAL EFFICIENTNET-B0 INITIALIZATION")
print("=" * 70)

# ------------------------------------------------------------
# 2. Build fresh ImageNet-pretrained EfficientNet-B0
# ------------------------------------------------------------

def build_final_efficientnet_b0():

    model = efficientnet_b0(
        weights=EfficientNet_B0_Weights.IMAGENET1K_V1
    )

    # Freeze all original parameters.
    for parameter in model.parameters():
        parameter.requires_grad = False

    # Replace ImageNet classifier with binary classifier.
    in_features = model.classifier[-1].in_features

    assert in_features == 1280

    model.classifier[-1] = nn.Linear(
        in_features,
        1
    )

    # Fine-tune the same components as Stage 5.
    for parameter in model.features[7].parameters():
        parameter.requires_grad = True

    for parameter in model.features[8].parameters():
        parameter.requires_grad = True

    for parameter in model.classifier[-1].parameters():
        parameter.requires_grad = True

    return model


final_model = build_final_efficientnet_b0().to(DEVICE)

print("Architecture: EfficientNet-B0")
print("Initialization: ImageNet IMAGENET1K_V1")
print("Classifier output features:",
      final_model.classifier[-1].out_features)

# ------------------------------------------------------------
# 3. Verify trainable parameter groups
# ------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in final_model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in final_model.parameters()
    if p.requires_grad
)

frozen_parameters = (
    total_parameters - trainable_parameters
)

assert total_parameters == 4_008_829, (
    f"Unexpected total parameter count: {total_parameters}"
)

assert trainable_parameters == 1_130_673, (
    f"Unexpected trainable count: {trainable_parameters}"
)

trainable_names = [
    name
    for name, parameter in final_model.named_parameters()
    if parameter.requires_grad
]

allowed_prefixes = (
    "features.7.",
    "features.8.",
    "classifier.1."
)

assert all(
    name.startswith(allowed_prefixes)
    for name in trainable_names
)

assert any(
    name.startswith("features.7.")
    for name in trainable_names
)

assert any(
    name.startswith("features.8.")
    for name in trainable_names
)

assert any(
    name.startswith("classifier.1.")
    for name in trainable_names
)

print("\n" + "=" * 70)
print("MODEL PARAMETER AUDIT")
print("=" * 70)

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Frozen parameters:", f"{frozen_parameters:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.2f}%"
)

# ------------------------------------------------------------
# 4. Preserve frozen batch-normalization statistics
# ------------------------------------------------------------

def set_final_efficientnet_training_mode(model):

    model.train()

    # Match the Stage 5 batch-normalization policy:
    # trainable affine parameters can receive gradients,
    # but stored running statistics do not update.
    for module in model.modules():

        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


set_final_efficientnet_training_mode(final_model)

batchnorm_modules = [
    module
    for module in final_model.modules()
    if isinstance(module, nn.modules.batchnorm._BatchNorm)
]

assert len(batchnorm_modules) == 49

assert all(
    not module.training
    for module in batchnorm_modules
)

assert final_model.training is True

assert final_model.features[7].training is True
assert final_model.features[8].training is True

print("\nBatchNorm modules:", len(batchnorm_modules))
print("BatchNorm running statistics frozen: True")

# ------------------------------------------------------------
# 5. Configure established class-weighted loss
# ------------------------------------------------------------

final_pos_weight_tensor = torch.tensor(
    [FINAL_POS_WEIGHT],
    dtype=torch.float32,
    device=DEVICE
)

final_criterion = nn.BCEWithLogitsLoss(
    pos_weight=final_pos_weight_tensor
)

assert np.isclose(
    final_pos_weight_tensor.item(),
    5178 / 12717,
    atol=1e-7
)

print("\nLoss: BCEWithLogitsLoss")
print(
    "Malignant-class pos_weight:",
    round(final_pos_weight_tensor.item(), 6)
)

# ------------------------------------------------------------
# 6. Configure AdamW with two learning rates
# ------------------------------------------------------------

feature_parameters = (
    list(final_model.features[7].parameters())
    + list(final_model.features[8].parameters())
)

classifier_parameters = list(
    final_model.classifier[-1].parameters()
)

assert all(
    parameter.requires_grad
    for parameter in feature_parameters
)

assert all(
    parameter.requires_grad
    for parameter in classifier_parameters
)

assert not (
    {id(p) for p in feature_parameters}
    & {id(p) for p in classifier_parameters}
)

final_optimizer = torch.optim.AdamW(
    [
        {
            "params": feature_parameters,
            "lr": FINAL_FEATURE_LR
        },
        {
            "params": classifier_parameters,
            "lr": FINAL_CLASSIFIER_LR
        }
    ],
    weight_decay=FINAL_WEIGHT_DECAY
)

optimizer_parameters = [
    parameter
    for group in final_optimizer.param_groups
    for parameter in group["params"]
]

assert sum(
    p.numel()
    for p in optimizer_parameters
) == trainable_parameters

assert len(
    {id(p) for p in optimizer_parameters}
) == len(optimizer_parameters)

assert np.isclose(
    final_optimizer.param_groups[0]["lr"],
    1e-4
)

assert np.isclose(
    final_optimizer.param_groups[1]["lr"],
    5e-4
)

assert all(
    np.isclose(group["weight_decay"], 1e-4)
    for group in final_optimizer.param_groups
)

print("\n" + "=" * 70)
print("OPTIMIZER CONFIGURATION")
print("=" * 70)

print("Optimizer: AdamW")
print("Features 7 and 8 LR:", FINAL_FEATURE_LR)
print("Classifier LR:", FINAL_CLASSIFIER_LR)
print("Weight decay:", FINAL_WEIGHT_DECAY)

# ------------------------------------------------------------
# 7. Prepare FP16 automatic mixed precision
# ------------------------------------------------------------

assert DEVICE.type == "cuda"

final_scaler = torch.amp.GradScaler("cuda")

assert final_scaler.is_enabled()

print("\nAutomatic mixed precision: CUDA FP16")
print("GradScaler enabled:", final_scaler.is_enabled())

# ------------------------------------------------------------
# 8. Lock the training configuration
# ------------------------------------------------------------

FINAL_TRAINING_CONFIG = {
    "architecture": "efficientnet_b0",
    "initial_weights": "IMAGENET1K_V1",
    "seed": FINAL_SEED,
    "epochs": FINAL_EPOCHS,
    "batch_size": FINAL_BATCH_SIZE,
    "development_patients": 49,
    "development_images": 1133,
    "development_patches": 17895,
    "source_mpp": float(SOURCE_MPP_BREAKHIS),
    "target_mpp": float(TARGET_MPP),
    "trainable_components": [
        "features.7",
        "features.8",
        "classifier.1"
    ],
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "optimizer": "AdamW",
    "feature_learning_rate": FINAL_FEATURE_LR,
    "classifier_learning_rate": FINAL_CLASSIFIER_LR,
    "weight_decay": FINAL_WEIGHT_DECAY,
    "loss": "BCEWithLogitsLoss",
    "positive_class_weight": float(
        final_pos_weight_tensor.item()
    ),
    "batchnorm_running_statistics": "frozen",
    "mixed_precision": "cuda_fp16",
    "checkpoint_selection": "final_epoch_5",
    "validation_dataset_used": False,
    "calibration_dataset_used": False,
    "test_dataset_used": False
}

# ------------------------------------------------------------
# 9. Final audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 6 — FINAL MODEL CONFIGURATION AUDIT")
print("=" * 70)

print("PASS: Fresh ImageNet-pretrained EfficientNet-B0 created.")
print("PASS: Binary classification head initialized.")
print("PASS: Total parameter count matches Stage 5.")
print("PASS: Trainable parameter count matches Stage 5.")
print("PASS: Only intended layers are trainable.")
print("PASS: All 49 BatchNorm modules frozen in eval mode.")
print("PASS: Full-Development positive-class weight applied.")
print("PASS: AdamW parameter groups verified.")
print("PASS: FP16 GradScaler initialized.")
print("PASS: Five-epoch training configuration locked.")
print("PASS: No fold checkpoint reused.")
print("PASS: No Calibration or Test data accessed.")
print("PASS: No training performed yet.")


STAGE 6 — FINAL EFFICIENTNET-B0 INITIALIZATION
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 107MB/s] 

Architecture: EfficientNet-B0
Initialization: ImageNet IMAGENET1K_V1
Classifier output features: 1

MODEL PARAMETER AUDIT
Total parameters: 4,008,829
Trainable parameters: 1,130,673
Frozen parameters: 2,878,156
Trainable percentage: 28.20%

BatchNorm modules: 49
BatchNorm running statistics frozen: True

Loss: BCEWithLogitsLoss
Malignant-class pos_weight: 0.407172

OPTIMIZER CONFIGURATION
Optimizer: AdamW
Features 7 and 8 LR: 0.0001
Classifier LR: 0.0005
Weight decay: 0.0001

Automatic mixed precision: CUDA FP16
GradScaler enabled: True

STAGE 6 — FINAL MODEL CONFIGURATION AUDIT
PASS: Fresh ImageNet-pretrained EfficientNet-B0 created.
PASS: Binary classification head initialized.
PASS: Total parameter count matches Stage 5.
PASS: Trainable parameter count matches Stage 5.
PASS: Only intended layers are trainable.
PASS: All 49 BatchNorm modules frozen in eval mode.
PASS: Full-Development positive-class weight applied.
PASS: AdamW parameter groups verified.
PASS: FP16 GradScaler initiali

In [10]:

# ============================================================
# Training Routine Restoration and Final Preflight
# ============================================================

import ast
import hashlib
import inspect
import json
import random

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Locate the completed Stage 5 EfficientNet notebook
# ------------------------------------------------------------

STAGE5_NOTEBOOK = (
    REPO_ROOT /
    "Notebooks/05_efficientnet_b0_formal_development.ipynb"
)

assert STAGE5_NOTEBOOK.is_file(), (
    "Stage 5 EfficientNet notebook not found. "
    f"Expected: {STAGE5_NOTEBOOK}"
)

with open(
    STAGE5_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    notebook = json.load(f)

target_function = "train_efficientnet_one_epoch"

matching_functions = []

for cell_number, cell in enumerate(
    notebook["cells"],
    start=1
):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")

    if isinstance(source, list):
        source = "".join(source)

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    for node in tree.body:

        if (
            isinstance(node, ast.FunctionDef)
            and node.name == target_function
        ):
            matching_functions.append(
                (cell_number, node)
            )

assert len(matching_functions) == 1, (
    "Expected exactly one original "
    "train_efficientnet_one_epoch definition; "
    f"found {len(matching_functions)}."
)

source_cell, training_node = matching_functions[0]

print("=" * 70)
print("STAGE 6 — TRAINING FUNCTION RESTORATION")
print("=" * 70)

print("Source:", STAGE5_NOTEBOOK.name)
print("Original cell:", source_cell)

# ------------------------------------------------------------
# 2. Restore the original training function
# ------------------------------------------------------------

# The Stage 5 routine may call this function by its
# original name. The implementation has already been
# verified in Cell 6.
set_efficientnet_training_mode = (
    set_final_efficientnet_training_mode
)

module = ast.Module(
    body=[training_node],
    type_ignores=[]
)

ast.fix_missing_locations(module)

exec(
    compile(
        module,
        str(STAGE5_NOTEBOOK),
        "exec"
    ),
    globals(),
    globals()
)

assert callable(train_efficientnet_one_epoch)

expected_arguments = [
    "model",
    "loader",
    "criterion",
    "optimizer",
    "scaler",
    "device",
    "epoch"
]

actual_arguments = list(
    inspect.signature(
        train_efficientnet_one_epoch
    ).parameters
)

assert actual_arguments == expected_arguments, (
    f"Unexpected training signature: {actual_arguments}"
)

training_source = ast.unparse(training_node)

training_source_hash = hashlib.sha256(
    training_source.encode("utf-8")
).hexdigest()

print("\nPASS: Original Stage 5 training function restored.")
print("Signature:", inspect.signature(
    train_efficientnet_one_epoch
))
print("Function SHA-256:", training_source_hash)

# Show the return contract for the next training cell.
return_expressions = [
    ast.unparse(node.value)
    if node.value is not None else "None"
    for node in ast.walk(training_node)
    if isinstance(node, ast.Return)
]

print("Original return expressions:", return_expressions)

# ------------------------------------------------------------
# 3. Recreate the DataLoader with a fresh generator
# ------------------------------------------------------------

# The earlier DataLoader was used to inspect a batch.
# A brand-new loader prevents that inspection from
# changing the training shuffle and worker seeds.

fresh_generator = torch.Generator()
fresh_generator.manual_seed(FINAL_SEED)

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=FINAL_BATCH_SIZE,
    shuffle=True,
    num_workers=FINAL_NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=FINAL_NUM_WORKERS > 0,
    worker_init_fn=seed_final_training_worker,
    generator=fresh_generator
)

assert len(final_train_loader) == 560

assert len(final_train_dataset) == 17895

# No iteration here: leave RNG at the beginning
# of the actual training sequence.

print("\nPASS: Fresh training DataLoader created.")
print("Batches per epoch:", len(final_train_loader))
print("Shuffle generator seed:", FINAL_SEED)

# ------------------------------------------------------------
# 4. Verify training has not yet started
# ------------------------------------------------------------

assert len(final_optimizer.state) == 0, (
    "Optimizer already contains training state. "
    "Do not restart initialization mid-training."
)

assert all(
    parameter.grad is None
    for parameter in final_model.parameters()
)

set_final_efficientnet_training_mode(final_model)

bn_modules = [
    module
    for module in final_model.modules()
    if isinstance(
        module,
        torch.nn.modules.batchnorm._BatchNorm
    )
]

assert len(bn_modules) == 49

assert all(
    not module.training
    for module in bn_modules
)

trainable_count = sum(
    parameter.numel()
    for parameter in final_model.parameters()
    if parameter.requires_grad
)

assert trainable_count == 1_130_673

print("PASS: Optimizer has no training state.")
print("PASS: Model has no existing gradients.")
print("PASS: All BatchNorm running statistics frozen.")

# ------------------------------------------------------------
# 5. Prepare checkpoint and results locations
# ------------------------------------------------------------

# Checkpoint weights remain outside the GitHub repo.
FINAL_CHECKPOINT_DIR = Path(
    "/content/CIS631_model_checkpoints/"
    "efficientnet_b0/final_development"
)

FINAL_RESULTS_DIR = (
    REPO_ROOT /
    "results/final_development/efficientnet_b0"
)

FINAL_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_CHECKPOINT_PATH = (
    FINAL_CHECKPOINT_DIR /
    "efficientnet_b0_final_epoch5.pt"
)

FINAL_HISTORY_PATH = (
    FINAL_RESULTS_DIR /
    "final_training_history.csv"
)

FINAL_CONFIG_PATH = (
    FINAL_RESULTS_DIR /
    "final_training_config.json"
)

assert not FINAL_CHECKPOINT_PATH.exists(), (
    "A final checkpoint already exists. "
    "Inspect it before starting another training run."
)

# ------------------------------------------------------------
# 6. Record source and dataset provenance
# ------------------------------------------------------------

manifest_sha256 = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

training_config = {
    **FINAL_TRAINING_CONFIG,
    "source_training_notebook": (
        str(STAGE5_NOTEBOOK.relative_to(REPO_ROOT))
    ),
    "source_training_function": target_function,
    "source_training_function_sha256": (
        training_source_hash
    ),
    "development_manifest": (
        str(manifest_path.relative_to(REPO_ROOT))
    ),
    "development_manifest_sha256": manifest_sha256,
    "training_batches_per_epoch": 560,
    "dataloader_workers": FINAL_NUM_WORKERS,
    "dataloader_seed": FINAL_SEED,
    "checkpoint_location": str(
        FINAL_CHECKPOINT_PATH
    ),
    "epoch_selection": (
        "fixed_epoch_5_no_validation_selection"
    )
}

with open(
    FINAL_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        training_config,
        f,
        indent=2
    )

assert FINAL_CONFIG_PATH.is_file()

print("\nResults directory:", FINAL_RESULTS_DIR)
print("Checkpoint directory:", FINAL_CHECKPOINT_DIR)
print("Development manifest SHA-256:", manifest_sha256)

# ------------------------------------------------------------
# 7. Final readiness audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 6 — TRAINING READINESS AUDIT")
print("=" * 70)

print("PASS: Stage 5 training function restored.")
print("PASS: Original function signature verified.")
print("PASS: Training source SHA-256 recorded.")
print("PASS: Fresh DataLoader seeded at 631.")
print("PASS: All 17,895 patches ready for training.")
print("PASS: Optimizer has not performed any updates.")
print("PASS: BatchNorm policy preserved.")
print("PASS: Final training configuration saved.")
print("PASS: Checkpoint storage prepared outside GitHub.")
print("PASS: Five-epoch training decision unchanged.")
print("PASS: No Calibration, Test or external data accessed.")
print("PASS: Ready to begin final training.")


STAGE 6 — TRAINING FUNCTION RESTORATION
Source: 05_efficientnet_b0_formal_development.ipynb
Original cell: 22

PASS: Original Stage 5 training function restored.
Signature: (model, loader, criterion, optimizer, scaler, device, epoch)
Function SHA-256: 6af08bbc2f34fa393f8c5dad2b10465d50acfad62f5894a9c803a4d1f5e78268
Original return expressions: ['total_loss / total_examples']

PASS: Fresh training DataLoader created.
Batches per epoch: 560
Shuffle generator seed: 631
PASS: Optimizer has no training state.
PASS: Model has no existing gradients.
PASS: All BatchNorm running statistics frozen.

Results directory: /content/breast-histology-calibration/results/final_development/efficientnet_b0
Checkpoint directory: /content/CIS631_model_checkpoints/efficientnet_b0/final_development
Development manifest SHA-256: 07cab77292894938a7ca7655239dcfec40e3a79e34b83042888b616da13d05b4

STAGE 6 — TRAINING READINESS AUDIT
PASS: Stage 5 training function restored.
PASS: Original function signature verifie

In [12]:

# ============================================================
# Final EfficientNet-B0: Train Epoch 1 of 5
# Dependency Fix, Safety Checks, Checkpoint and Backup
# ============================================================

import os
import json
import time
import random
import hashlib
import zipfile
import builtins
import dis

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from google.colab import files
from IPython.display import display

# ------------------------------------------------------------
# 1. Restore Stage 5 training-function dependencies
# ------------------------------------------------------------

# The original Stage 5 training function references EPOCHS.
# Stage 6 uses FINAL_EPOCHS, so restore the expected alias.

EPOCHS = FINAL_EPOCHS

# Preserve the original BatchNorm training policy.
set_efficientnet_training_mode = (
    set_final_efficientnet_training_mode
)

assert EPOCHS == 5
assert FINAL_SEED == 631
assert FINAL_BATCH_SIZE == 32

# Check for any other unresolved global dependencies.
missing_globals = sorted({
    instruction.argval
    for instruction in dis.get_instructions(
        train_efficientnet_one_epoch
    )
    if instruction.opname == "LOAD_GLOBAL"
    and instruction.argval not in globals()
    and instruction.argval not in vars(builtins)
})

assert not missing_globals, (
    "Additional training dependencies are missing: "
    f"{missing_globals}"
)

print("=" * 72)
print("STAGE 6 — TRAINING DEPENDENCY VERIFICATION")
print("=" * 72)

print("Original EPOCHS variable:", EPOCHS)
print("Missing training dependencies:", missing_globals)

print("PASS: Original Stage 5 training dependencies available.")

# ------------------------------------------------------------
# 2. Verify it is safe to restart Epoch 1
# ------------------------------------------------------------

assert len(final_optimizer.state) == 0, (
    "Optimizer already contains training state. "
    "Do not restart Epoch 1 without restoring "
    "a clean model."
)

assert not FINAL_HISTORY_PATH.exists(), (
    "A training history already exists. "
    "Do not overwrite completed progress."
)

assert not FINAL_CHECKPOINT_PATH.exists(), (
    "Final Epoch 5 checkpoint already exists."
)

EPOCH = 1

epoch_checkpoint_path = (
    FINAL_CHECKPOINT_DIR /
    "efficientnet_b0_final_epoch1_resume.pt"
)

assert not epoch_checkpoint_path.exists(), (
    "Epoch 1 checkpoint already exists. "
    "Do not overwrite it."
)

# The previous failed Cell 8 captured initial weights.
# Compare against them before taking a new snapshot.

if "initial_classifier_weight" in globals():

    assert torch.equal(
        final_model.classifier[-1].weight.detach(),
        initial_classifier_weight
    ), (
        "Classifier weights changed during the "
        "previous failed attempt. Stop training."
    )

    print("PASS: Classifier unchanged since failed attempt.")

if "initial_bn_statistics" in globals():

    previous_bn_modules = [
        module
        for module in final_model.modules()
        if isinstance(
            module,
            torch.nn.modules.batchnorm._BatchNorm
        )
    ]

    assert len(previous_bn_modules) == len(
        initial_bn_statistics
    )

    for module, original in zip(
        previous_bn_modules,
        initial_bn_statistics
    ):

        assert torch.equal(
            module.running_mean,
            original["running_mean"]
        )

        assert torch.equal(
            module.running_var,
            original["running_var"]
        )

        assert torch.equal(
            module.num_batches_tracked,
            original["num_batches_tracked"]
        )

    print("PASS: BatchNorm statistics unchanged.")

# Clear any gradients from the interrupted call.
final_optimizer.zero_grad(set_to_none=True)

assert len(final_optimizer.state) == 0

print("PASS: No optimizer updates occurred.")
print("PASS: Safe to start Epoch 1.")

# ------------------------------------------------------------
# 3. Recreate the training DataLoader
# ------------------------------------------------------------

# The earlier DataLoader was used for inspection and
# may have started persistent workers.
#
# A fresh generator and DataLoader ensure that the
# earlier inspection does not affect training shuffle.

fresh_generator = torch.Generator()
fresh_generator.manual_seed(FINAL_SEED)

final_train_loader = DataLoader(
    dataset=final_train_dataset,
    batch_size=FINAL_BATCH_SIZE,
    shuffle=True,
    num_workers=FINAL_NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=FINAL_NUM_WORKERS > 0,
    worker_init_fn=seed_final_training_worker,
    generator=fresh_generator
)

assert len(final_train_dataset) == 17895
assert len(final_train_loader) == 560

print("\nPASS: Fresh training DataLoader created.")
print("Training patches:", len(final_train_dataset))
print("Batches per epoch:", len(final_train_loader))
print("DataLoader seed:", FINAL_SEED)

# ------------------------------------------------------------
# 4. Capture initial model state
# ------------------------------------------------------------

initial_classifier_weight = (
    final_model.classifier[-1]
    .weight.detach().clone()
)

set_final_efficientnet_training_mode(
    final_model
)

bn_modules = [
    module
    for module in final_model.modules()
    if isinstance(
        module,
        torch.nn.modules.batchnorm._BatchNorm
    )
]

assert len(bn_modules) == 49
assert all(not module.training for module in bn_modules)

initial_bn_statistics = [
    {
        "running_mean":
            module.running_mean.detach().clone(),

        "running_var":
            module.running_var.detach().clone(),

        "num_batches_tracked":
            module.num_batches_tracked.detach().clone()
    }
    for module in bn_modules
]

print("\nPASS: Initial classifier weights recorded.")
print("PASS: All 49 BatchNorm modules in eval mode.")

# ------------------------------------------------------------
# 5. Train Epoch 1 using the restored Stage 5 function
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("STAGE 6 — FINAL EFFICIENTNET-B0 TRAINING")
print("=" * 72)

print("Epoch:", EPOCH, "of", FINAL_EPOCHS)
print("Development patients:", 49)
print("Training patches:", len(final_train_dataset))
print("Training batches:", len(final_train_loader))
print("Device:", DEVICE)
print("Positive-class weight:", round(FINAL_POS_WEIGHT, 6))

print("\nStarting Epoch 1...")

start_time = time.perf_counter()

epoch_loss = train_efficientnet_one_epoch(
    model=final_model,
    loader=final_train_loader,
    criterion=final_criterion,
    optimizer=final_optimizer,
    scaler=final_scaler,
    device=DEVICE,
    epoch=EPOCH
)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

elapsed_seconds = time.perf_counter() - start_time

epoch_loss = float(epoch_loss)

assert np.isfinite(epoch_loss)
assert epoch_loss >= 0

print("\nEpoch 1 training loss:", round(epoch_loss, 6))
print("Elapsed seconds:", round(elapsed_seconds, 2))

# ------------------------------------------------------------
# 6. Verify parameter updates and frozen BatchNorm
# ------------------------------------------------------------

classifier_change = float(
    (
        final_model.classifier[-1].weight.detach()
        - initial_classifier_weight
    ).abs().mean().item()
)

assert classifier_change > 0, (
    "Classifier weights did not change."
)

assert len(final_optimizer.state) > 0, (
    "Optimizer state is empty after training."
)

for module, original in zip(
    bn_modules,
    initial_bn_statistics
):

    assert not module.training

    assert torch.equal(
        module.running_mean,
        original["running_mean"]
    )

    assert torch.equal(
        module.running_var,
        original["running_var"]
    )

    assert torch.equal(
        module.num_batches_tracked,
        original["num_batches_tracked"]
    )

print("\nPASS: Optimizer updates verified.")
print("PASS: Classifier weights changed.")
print("PASS: BatchNorm running statistics remained frozen.")

# ------------------------------------------------------------
# 7. Save Epoch 1 training history
# ------------------------------------------------------------

epoch_record = {
    "epoch": EPOCH,
    "training_loss": epoch_loss,
    "training_patches": 17895,
    "batches": len(final_train_loader),
    "batch_size": FINAL_BATCH_SIZE,
    "elapsed_seconds": elapsed_seconds,
    "validation_performed": False
}

final_training_history = pd.DataFrame([
    epoch_record
])

final_training_history.to_csv(
    FINAL_HISTORY_PATH,
    index=False
)

assert FINAL_HISTORY_PATH.is_file()

print("PASS: Epoch 1 training history saved.")

# ------------------------------------------------------------
# 8. Save complete resumable training checkpoint
# ------------------------------------------------------------

checkpoint_payload = {
    "architecture": "efficientnet_b0",
    "epoch_completed": EPOCH,
    "total_epochs": FINAL_EPOCHS,
    "training_loss": epoch_loss,

    "model_state_dict":
        final_model.state_dict(),

    "optimizer_state_dict":
        final_optimizer.state_dict(),

    "grad_scaler_state_dict":
        final_scaler.state_dict(),

    "dataloader_generator_state":
        final_train_loader.generator.get_state(),

    "torch_rng_state":
        torch.get_rng_state(),

    "cuda_rng_state_all":
        torch.cuda.get_rng_state_all(),

    "numpy_rng_state":
        np.random.get_state(),

    "python_rng_state":
        random.getstate(),

    "training_config": training_config,

    "validation_performed": False,
    "calibration_data_used": False,
    "test_data_used": False
}

temporary_checkpoint = (
    epoch_checkpoint_path.with_suffix(".pt.tmp")
)

torch.save(
    checkpoint_payload,
    temporary_checkpoint
)

# Replace the temporary file only after a complete write.
os.replace(
    temporary_checkpoint,
    epoch_checkpoint_path
)

assert epoch_checkpoint_path.is_file()
assert epoch_checkpoint_path.stat().st_size > 0

print("PASS: Complete Epoch 1 checkpoint saved.")

# ------------------------------------------------------------
# 9. Independently reload and verify checkpoint
# ------------------------------------------------------------

# The checkpoint is from our own trusted Colab runtime.
saved_checkpoint = torch.load(
    epoch_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

assert saved_checkpoint["epoch_completed"] == 1
assert saved_checkpoint["total_epochs"] == 5

assert np.isclose(
    saved_checkpoint["training_loss"],
    epoch_loss,
    atol=1e-12
)

assert set(
    saved_checkpoint["model_state_dict"].keys()
) == set(
    final_model.state_dict().keys()
)

assert len(
    saved_checkpoint["optimizer_state_dict"]["state"]
) > 0

assert torch.equal(
    saved_checkpoint[
        "model_state_dict"
    ]["classifier.1.weight"],
    final_model.classifier[-1].weight.detach().cpu()
)

assert "grad_scaler_state_dict" in saved_checkpoint
assert "torch_rng_state" in saved_checkpoint
assert "cuda_rng_state_all" in saved_checkpoint
assert "numpy_rng_state" in saved_checkpoint
assert "python_rng_state" in saved_checkpoint

del saved_checkpoint

print("PASS: Checkpoint independently reloaded.")
print("PASS: Model, optimizer and RNG state verified.")

# ------------------------------------------------------------
# 10. Calculate checkpoint SHA-256
# ------------------------------------------------------------

def calculate_file_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


checkpoint_sha256 = calculate_file_sha256(
    epoch_checkpoint_path
)

print("PASS: Checkpoint SHA-256 calculated.")

# ------------------------------------------------------------
# 11. Save progress and integrity records
# ------------------------------------------------------------

progress_path = (
    FINAL_RESULTS_DIR /
    "final_training_progress.json"
)

integrity_path = (
    FINAL_RESULTS_DIR /
    "epoch1_checkpoint_integrity.json"
)

progress_record = {
    "architecture": "efficientnet_b0",
    "epochs_completed": 1,
    "epochs_planned": 5,
    "latest_training_loss": epoch_loss,
    "latest_checkpoint": str(epoch_checkpoint_path),
    "validation_performed": False,
    "training_complete": False
}

integrity_record = {
    "epoch": 1,
    "checkpoint_filename": epoch_checkpoint_path.name,
    "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_size_bytes": (
        epoch_checkpoint_path.stat().st_size
    ),
    "checkpoint_reload_passed": True,
    "optimizer_state_saved": True,
    "grad_scaler_state_saved": True,
    "rng_states_saved": True,
    "batchnorm_running_statistics_unchanged": True,
    "source_training_function_sha256":
        training_source_hash,
    "development_manifest_sha256":
        manifest_sha256
}

with open(
    progress_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(progress_record, f, indent=2)

with open(
    integrity_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(integrity_record, f, indent=2)

assert progress_path.is_file()
assert integrity_path.is_file()

print("PASS: Progress and integrity records saved.")

# ------------------------------------------------------------
# 12. Create downloadable Epoch 1 backup
# ------------------------------------------------------------

backup_path = Path(
    "/content/"
    "CIS631_EfficientNetB0_Final_Epoch1_Backup.zip"
)

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    # Model weights and optimizer state: outside GitHub.
    zf.write(
        epoch_checkpoint_path,
        arcname=(
            "checkpoints/"
            + epoch_checkpoint_path.name
        )
    )

    # Small research metadata files.
    for path in [
        FINAL_HISTORY_PATH,
        FINAL_CONFIG_PATH,
        progress_path,
        integrity_path
    ]:

        assert path.is_file()

        zf.write(
            path,
            arcname=str(
                Path("results/final_development/")
                / "efficientnet_b0"
                / path.name
            )
        )

with zipfile.ZipFile(backup_path) as zf:

    assert zf.testzip() is None
    assert len(zf.namelist()) == 5

print("PASS: Epoch 1 backup ZIP verified.")

# ------------------------------------------------------------
# 13. Display training results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FINAL TRAINING HISTORY — EPOCH 1")
print("=" * 72)

display(
    final_training_history.round(6)
)

print(
    "\nClassifier mean absolute weight change:",
    round(classifier_change, 8)
)

print("Checkpoint SHA-256:", checkpoint_sha256)
print("Checkpoint location:", epoch_checkpoint_path)
print("Backup ZIP:", backup_path)

# ------------------------------------------------------------
# 14. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("STAGE 6 — EPOCH 1 COMPLETION AUDIT")
print("=" * 72)

print("PASS: Epoch 1 of 5 completed.")
print("PASS: Finite training loss recorded.")
print("PASS: Classifier parameters updated.")
print("PASS: Optimizer state populated.")
print("PASS: BatchNorm statistics remained frozen.")
print("PASS: Complete training checkpoint saved.")
print("PASS: Checkpoint independently reloaded.")
print("PASS: Checkpoint SHA-256 recorded.")
print("PASS: Training history saved.")
print("PASS: Backup ZIP created and verified.")
print("PASS: No validation-based epoch selection.")
print("PASS: Calibration and Test data untouched.")

# ------------------------------------------------------------
# 15. Download Epoch 1 backup
# ------------------------------------------------------------

files.download(str(backup_path))


STAGE 6 — TRAINING DEPENDENCY VERIFICATION
Original EPOCHS variable: 5
Missing training dependencies: []
PASS: Original Stage 5 training dependencies available.
PASS: Classifier unchanged since failed attempt.
PASS: BatchNorm statistics unchanged.
PASS: No optimizer updates occurred.
PASS: Safe to start Epoch 1.

PASS: Fresh training DataLoader created.
Training patches: 17895
Batches per epoch: 560
DataLoader seed: 631

PASS: Initial classifier weights recorded.
PASS: All 49 BatchNorm modules in eval mode.

STAGE 6 — FINAL EFFICIENTNET-B0 TRAINING
Epoch: 1 of 5
Development patients: 49
Training patches: 17895
Training batches: 560
Device: cuda
Positive-class weight: 0.407172

Starting Epoch 1...


Epoch 1/5 training:   0%|          | 0/560 [00:00<?, ?it/s]


Epoch 1 training loss: 0.089836
Elapsed seconds: 344.94

PASS: Optimizer updates verified.
PASS: Classifier weights changed.
PASS: BatchNorm running statistics remained frozen.
PASS: Epoch 1 training history saved.
PASS: Complete Epoch 1 checkpoint saved.
PASS: Checkpoint independently reloaded.
PASS: Model, optimizer and RNG state verified.
PASS: Checkpoint SHA-256 calculated.
PASS: Progress and integrity records saved.
PASS: Epoch 1 backup ZIP verified.

FINAL TRAINING HISTORY — EPOCH 1


,epoch,training_loss,training_patches,batches,batch_size,elapsed_seconds,validation_performed
0,1,0.089836,17895,560,32,344.940912,False



Classifier mean absolute weight change: 0.01387459
Checkpoint SHA-256: a61ab8368c91a654c738030dd202630fe041d5b00ba0b8e1e7658092af92538e
Checkpoint location: /content/CIS631_model_checkpoints/efficientnet_b0/final_development/efficientnet_b0_final_epoch1_resume.pt
Backup ZIP: /content/CIS631_EfficientNetB0_Final_Epoch1_Backup.zip

STAGE 6 — EPOCH 1 COMPLETION AUDIT
PASS: Epoch 1 of 5 completed.
PASS: Finite training loss recorded.
PASS: Classifier parameters updated.
PASS: Optimizer state populated.
PASS: BatchNorm statistics remained frozen.
PASS: Complete training checkpoint saved.
PASS: Checkpoint independently reloaded.
PASS: Checkpoint SHA-256 recorded.
PASS: Training history saved.
PASS: Backup ZIP created and verified.
PASS: No validation-based epoch selection.
PASS: Calibration and Test data untouched.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:

# ============================================================
# Restore Last Verified Checkpoint and Finish Through Epoch 5
# ============================================================

# ------------------------------------------------------------
# 1. Locked training configuration
# ------------------------------------------------------------

EPOCHS = FINAL_EPOCHS

set_efficientnet_training_mode = (
    set_final_efficientnet_training_mode
)

assert EPOCHS == 5
assert FINAL_SEED == 631
assert FINAL_BATCH_SIZE == 32
assert len(final_train_dataset) == 17895

assert FINAL_HISTORY_PATH.is_file()
assert FINAL_CONFIG_PATH.is_file()

FINAL_CHECKPOINT_DIR.mkdir(
    parents=True, exist_ok=True
)

FINAL_RESULTS_DIR.mkdir(
    parents=True, exist_ok=True
)

progress_path = (
    FINAL_RESULTS_DIR / "final_training_progress.json"
)


def checkpoint_path_for_epoch(epoch):
    if epoch == 5:
        return FINAL_CHECKPOINT_PATH

    return (
        FINAL_CHECKPOINT_DIR /
        f"efficientnet_b0_final_epoch{epoch}_resume.pt"
    )


def integrity_path_for_epoch(epoch):
    return (
        FINAL_RESULTS_DIR /
        f"epoch{epoch}_checkpoint_integrity.json"
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_json(data, path):
    temp = path.with_suffix(path.suffix + ".tmp")

    with open(temp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    os.replace(temp, path)


def verify_states(a, b):
    if torch.is_tensor(a) and torch.is_tensor(b):
        return torch.equal(
            a.detach().cpu(),
            b.detach().cpu()
        )

    if isinstance(a, dict) and isinstance(b, dict):
        return (
            a.keys() == b.keys()
            and all(
                verify_states(a[k], b[k])
                for k in a
            )
        )

    if isinstance(a, (list, tuple)) and isinstance(
        b, (list, tuple)
    ):
        return (
            len(a) == len(b)
            and all(
                verify_states(x, y)
                for x, y in zip(a, b)
            )
        )

    return a == b


# ------------------------------------------------------------
# 2. Identify the latest fully verified checkpoint
# ------------------------------------------------------------

verified_epochs = []

for epoch in range(1, 6):

    cp = checkpoint_path_for_epoch(epoch)
    ip = integrity_path_for_epoch(epoch)

    if not cp.exists() and not ip.exists():
        continue

    if cp.exists() != ip.exists():
        raise RuntimeError(
            f"Epoch {epoch} has an incomplete checkpoint/"
            "integrity pair. Do not overwrite it. "
            "Share the file status before continuing."
        )

    with open(ip, encoding="utf-8") as f:
        integrity = json.load(f)

    assert integrity["epoch"] == epoch
    assert integrity["checkpoint_filename"] == cp.name

    assert sha256_file(cp) == (
        integrity["checkpoint_sha256"]
    ), f"Checkpoint hash mismatch at Epoch {epoch}"

    verified_epochs.append(epoch)

assert verified_epochs, (
    "No verified checkpoint found."
)

assert verified_epochs == list(
    range(1, max(verified_epochs) + 1)
), "Completed epoch checkpoints are not sequential."

last_completed_epoch = max(verified_epochs)

assert last_completed_epoch < 5, (
    "Epoch 5 is already complete. "
    "Do not train it again."
)

history = pd.read_csv(FINAL_HISTORY_PATH)

assert history["epoch"].tolist() == list(
    range(1, last_completed_epoch + 1)
), (
    "Training history does not match verified "
    "checkpoints. Stop rather than overwrite progress."
)

with open(progress_path, encoding="utf-8") as f:
    progress = json.load(f)

assert progress["epochs_completed"] == (
    last_completed_epoch
)

print("=" * 72)
print("STAGE 6 — INTERRUPTION RECOVERY")
print("=" * 72)

print("Verified epochs:", verified_epochs)
print("Restoring Epoch:", last_completed_epoch)
print(
    "Remaining epochs:",
    list(range(last_completed_epoch + 1, 6))
)

print("PASS: Saved checkpoints verified by SHA-256.")
print("PASS: Training history matches checkpoints.")

# ------------------------------------------------------------
# 3. Restore complete model and optimizer state
# ------------------------------------------------------------

restore_checkpoint = checkpoint_path_for_epoch(
    last_completed_epoch
)

# Trusted checkpoint created by our own Colab training.
checkpoint = torch.load(
    restore_checkpoint,
    map_location="cpu",
    weights_only=False
)

assert checkpoint["epoch_completed"] == (
    last_completed_epoch
)

assert checkpoint["total_epochs"] == 5
assert checkpoint["architecture"] == "efficientnet_b0"

# Replace any weights partially updated by interruption.
final_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

final_optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

final_scaler.load_state_dict(
    checkpoint["grad_scaler_state_dict"]
)

# Clear any incomplete backward-pass gradients.
final_optimizer.zero_grad(set_to_none=True)

set_final_efficientnet_training_mode(final_model)

assert verify_states(
    checkpoint["model_state_dict"],
    final_model.state_dict()
)

assert verify_states(
    checkpoint["optimizer_state_dict"],
    final_optimizer.state_dict()
)

assert verify_states(
    checkpoint["grad_scaler_state_dict"],
    final_scaler.state_dict()
)

print("\nPASS: Model weights restored.")
print("PASS: Optimizer state restored.")
print("PASS: Gradient scaler restored.")
print("PASS: Interrupted gradients cleared.")

# ------------------------------------------------------------
# 4. Recreate the DataLoader and restore RNG states
# ------------------------------------------------------------

# Persistent worker RNG states are not stored in the
# checkpoint. Newly launched workers may therefore
# generate a different augmentation sequence.

old_iterator = getattr(
    final_train_loader, "_iterator", None
)

if old_iterator is not None:
    try:
        old_iterator._shutdown_workers()
    except Exception as error:
        print(
            "Worker shutdown warning:",
            type(error).__name__
        )

del old_iterator

restored_generator = torch.Generator()

restored_generator.set_state(
    checkpoint["dataloader_generator_state"]
)

final_train_loader = DataLoader(
    dataset=final_train_dataset,
    batch_size=FINAL_BATCH_SIZE,
    shuffle=True,
    num_workers=FINAL_NUM_WORKERS,
    pin_memory=True,
    drop_last=False,
    persistent_workers=FINAL_NUM_WORKERS > 0,
    worker_init_fn=seed_final_training_worker,
    generator=restored_generator
)

assert len(final_train_loader) == 560

# Restore the saved main-process RNG states.
torch.set_rng_state(checkpoint["torch_rng_state"])

torch.cuda.set_rng_state_all(
    checkpoint["cuda_rng_state_all"]
)

np.random.set_state(
    checkpoint["numpy_rng_state"]
)

random.setstate(
    checkpoint["python_rng_state"]
)

assert torch.equal(
    final_train_loader.generator.get_state(),
    checkpoint["dataloader_generator_state"]
)

del checkpoint

print("PASS: DataLoader recreated.")
print("PASS: DataLoader generator state restored.")
print("PASS: Main-process random states restored.")

# ------------------------------------------------------------
# 5. Continue all remaining epochs
# ------------------------------------------------------------

for epoch in range(last_completed_epoch + 1, 6):

    print("\n" + "=" * 72)
    print(f"FINAL EFFICIENTNET-B0 — EPOCH {epoch}/5")
    print("=" * 72)

    cp = checkpoint_path_for_epoch(epoch)
    ip = integrity_path_for_epoch(epoch)

    assert not cp.exists()
    assert not ip.exists()

    set_final_efficientnet_training_mode(final_model)

    # Record state before the epoch.
    classifier_before = (
        final_model.classifier[-1]
        .weight.detach().clone()
    )

    bn_modules = [
        m for m in final_model.modules()
        if isinstance(
            m, torch.nn.modules.batchnorm._BatchNorm
        )
    ]

    assert len(bn_modules) == 49
    assert all(not m.training for m in bn_modules)

    bn_before = [
        (
            m.running_mean.detach().clone(),
            m.running_var.detach().clone(),
            m.num_batches_tracked.detach().clone()
        )
        for m in bn_modules
    ]

    # ---------------------------------------------
    # Train one epoch
    # ---------------------------------------------

    start = time.perf_counter()

    loss = train_efficientnet_one_epoch(
        model=final_model,
        loader=final_train_loader,
        criterion=final_criterion,
        optimizer=final_optimizer,
        scaler=final_scaler,
        device=DEVICE,
        epoch=epoch
    )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    loss = float(loss)

    assert np.isfinite(loss)
    assert loss >= 0

    classifier_change = float(
        (
            final_model.classifier[-1].weight.detach()
            - classifier_before
        ).abs().mean().item()
    )

    assert classifier_change > 0

    for module, original in zip(
        bn_modules, bn_before
    ):
        assert not module.training

        assert torch.equal(
            module.running_mean,
            original[0]
        )

        assert torch.equal(
            module.running_var,
            original[1]
        )

        assert torch.equal(
            module.num_batches_tracked,
            original[2]
        )

    print(f"\nEpoch {epoch} loss: {loss:.6f}")
    print("Classifier change:", round(classifier_change, 8))
    print("PASS: BatchNorm running statistics unchanged.")

    # ---------------------------------------------
    # Build updated cumulative history
    # ---------------------------------------------

    epoch_row = {
        "epoch": epoch,
        "training_loss": loss,
        "training_patches": 17895,
        "batches": 560,
        "batch_size": FINAL_BATCH_SIZE,
        "elapsed_seconds": elapsed,
        "validation_performed": False
    }

    updated_history = pd.concat(
        [history, pd.DataFrame([epoch_row])],
        ignore_index=True
    )

    assert updated_history["epoch"].tolist() == list(
        range(1, epoch + 1)
    )

    # ---------------------------------------------
    # Save complete checkpoint atomically
    # ---------------------------------------------

    payload = {
        "architecture": "efficientnet_b0",
        "epoch_completed": epoch,
        "total_epochs": 5,
        "training_loss": loss,
        "model_state_dict":
            final_model.state_dict(),
        "optimizer_state_dict":
            final_optimizer.state_dict(),
        "grad_scaler_state_dict":
            final_scaler.state_dict(),
        "dataloader_generator_state":
            final_train_loader.generator.get_state(),
        "torch_rng_state":
            torch.get_rng_state(),
        "cuda_rng_state_all":
            torch.cuda.get_rng_state_all(),
        "numpy_rng_state":
            np.random.get_state(),
        "python_rng_state":
            random.getstate(),
        "training_config": training_config,
        "training_history":
            updated_history.to_dict("records"),
        "validation_performed": False,
        "calibration_data_used": False,
        "test_data_used": False
    }

    temporary_cp = cp.with_suffix(".pt.tmp")

    torch.save(payload, temporary_cp)
    os.replace(temporary_cp, cp)

    assert cp.is_file()
    assert cp.stat().st_size > 0

    # ---------------------------------------------
    # Independently reload and verify
    # ---------------------------------------------

    saved = torch.load(
        cp,
        map_location="cpu",
        weights_only=False
    )

    assert saved["epoch_completed"] == epoch

    assert np.isclose(
        saved["training_loss"],
        loss,
        atol=1e-12
    )

    assert verify_states(
        saved["model_state_dict"],
        final_model.state_dict()
    )

    assert verify_states(
        saved["optimizer_state_dict"],
        final_optimizer.state_dict()
    )

    assert verify_states(
        saved["grad_scaler_state_dict"],
        final_scaler.state_dict()
    )

    del saved

    cp_hash = sha256_file(cp)

    # ---------------------------------------------
    # Save integrity record, history and progress
    # ---------------------------------------------

    integrity = {
        "epoch": epoch,
        "checkpoint_filename": cp.name,
        "checkpoint_sha256": cp_hash,
        "checkpoint_size_bytes": cp.stat().st_size,
        "checkpoint_reload_passed": True,
        "optimizer_state_saved": True,
        "grad_scaler_state_saved": True,
        "rng_states_saved": True,
        "batchnorm_running_statistics_unchanged": True,
        "source_training_function_sha256":
            training_source_hash,
        "development_manifest_sha256":
            manifest_sha256
    }

    atomic_json(integrity, ip)

    temp_history = FINAL_HISTORY_PATH.with_suffix(
        ".csv.tmp"
    )

    updated_history.to_csv(
        temp_history,
        index=False
    )

    os.replace(temp_history, FINAL_HISTORY_PATH)

    progress = {
        "architecture": "efficientnet_b0",
        "epochs_completed": epoch,
        "epochs_planned": 5,
        "latest_training_loss": loss,
        "latest_checkpoint": str(cp),
        "validation_performed": False,
        "training_complete": epoch == 5
    }

    atomic_json(progress, progress_path)

    history = updated_history

    print(f"PASS: Epoch {epoch} checkpoint verified.")
    print(f"PASS: Epoch {epoch} history saved.")
    print("Checkpoint SHA-256:", cp_hash)

# ------------------------------------------------------------
# 6. Final five-epoch verification
# ------------------------------------------------------------

assert history["epoch"].tolist() == [1, 2, 3, 4, 5]

assert len(history) == 5

assert np.isfinite(
    history["training_loss"].to_numpy()
).all()

assert FINAL_CHECKPOINT_PATH.is_file()

for epoch in range(1, 6):

    cp = checkpoint_path_for_epoch(epoch)
    ip = integrity_path_for_epoch(epoch)

    assert cp.is_file()
    assert ip.is_file()

    with open(ip, encoding="utf-8") as f:
        integrity = json.load(f)

    assert sha256_file(cp) == (
        integrity["checkpoint_sha256"]
    )

print("\n" + "=" * 72)
print("FINAL DEVELOPMENT TRAINING HISTORY")
print("=" * 72)

display(history.round(6))

# ------------------------------------------------------------
# 7. Final completion record
# ------------------------------------------------------------

final_hash = sha256_file(FINAL_CHECKPOINT_PATH)

completion_path = (
    FINAL_RESULTS_DIR /
    "final_training_completion.json"
)

completion_record = {
    "architecture": "efficientnet_b0",
    "training_completed": True,
    "epochs_completed": 5,
    "checkpoint_selected": "fixed_epoch_5",
    "checkpoint_filename":
        FINAL_CHECKPOINT_PATH.name,
    "checkpoint_sha256": final_hash,
    "development_patients": 49,
    "development_images": 1133,
    "development_patches": 17895,
    "resumed_from_verified_epoch": (
        last_completed_epoch
    ),
    "persistent_worker_rng_exactly_restored": False,
    "validation_used": False,
    "calibration_used": False,
    "test_used": False,
    "external_data_used": False
}

atomic_json(completion_record, completion_path)

# ------------------------------------------------------------
# 8. Create final downloadable backup
# ------------------------------------------------------------

backup_path = Path(
    "/content/CIS631_EfficientNetB0_Final_Stage6_Backup.zip"
)

with zipfile.ZipFile(
    backup_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    # Final weights remain outside GitHub.
    zf.write(
        FINAL_CHECKPOINT_PATH,
        arcname=(
            "checkpoints/"
            + FINAL_CHECKPOINT_PATH.name
        )
    )

    metadata_files = [
        FINAL_HISTORY_PATH,
        FINAL_CONFIG_PATH,
        progress_path,
        completion_path
    ]

    metadata_files += [
        integrity_path_for_epoch(epoch)
        for epoch in range(1, 6)
    ]

    for path in metadata_files:

        assert path.is_file()

        zf.write(
            path,
            arcname=str(
                Path(
                    "results/final_development/efficientnet_b0"
                ) / path.name
            )
        )

with zipfile.ZipFile(backup_path) as zf:

    assert zf.testzip() is None
    assert len(zf.namelist()) == 10

print("\n" + "=" * 72)
print("STAGE 6 — FINAL COMPLETION AUDIT")
print("=" * 72)

print("PASS: Training resumed from verified checkpoint.")
print("PASS: All five training epochs completed.")
print("PASS: Final Epoch 5 checkpoint saved.")
print("PASS: All five checkpoint SHA-256 hashes verified.")
print("PASS: Cumulative training history verified.")
print("PASS: Optimizer and gradient-scaler states preserved.")
print("PASS: BatchNorm running statistics frozen.")
print("PASS: Final training completion record saved.")
print("PASS: Final backup ZIP verified.")
print("PASS: No validation-based checkpoint selection.")
print("PASS: Calibration, Test and external data untouched.")

print("\nStage 6 final training: COMPLETE")
print("Final checkpoint:", FINAL_CHECKPOINT_PATH)
print("Final SHA-256:", final_hash)
print("Backup:", backup_path)

files.download(str(backup_path))


STAGE 6 — INTERRUPTION RECOVERY
Verified epochs: [1]
Restoring Epoch: 1
Remaining epochs: [2, 3, 4, 5]
PASS: Saved checkpoints verified by SHA-256.
PASS: Training history matches checkpoints.

PASS: Model weights restored.
PASS: Optimizer state restored.
PASS: Gradient scaler restored.
PASS: Interrupted gradients cleared.
PASS: DataLoader recreated.
PASS: DataLoader generator state restored.
PASS: Main-process random states restored.

FINAL EFFICIENTNET-B0 — EPOCH 2/5


Epoch 2/5 training:   0%|          | 0/560 [00:00<?, ?it/s]


Epoch 2 loss: 0.035103
Classifier change: 0.00650403
PASS: BatchNorm running statistics unchanged.
PASS: Epoch 2 checkpoint verified.
PASS: Epoch 2 history saved.
Checkpoint SHA-256: 63ec23f7b9349eb8a20797ab11ebd8c934ca8f93af1f325bb7d4eaa92cd8c87b

FINAL EFFICIENTNET-B0 — EPOCH 3/5


Epoch 3/5 training:   0%|          | 0/560 [00:00<?, ?it/s]


Epoch 3 loss: 0.023573
Classifier change: 0.00542161
PASS: BatchNorm running statistics unchanged.
PASS: Epoch 3 checkpoint verified.
PASS: Epoch 3 history saved.
Checkpoint SHA-256: b788d7703545825611a8f2a2b5cd02996300b812a485791df60b5d956667f86b

FINAL EFFICIENTNET-B0 — EPOCH 4/5


...


Epoch 5 loss: 0.012382
Classifier change: 0.0046254
PASS: BatchNorm running statistics unchanged.
PASS: Epoch 5 checkpoint verified.
PASS: Epoch 5 history saved.
Checkpoint SHA-256: 84ea0871a309f3d68b8958b68b6da4f01adb31d3109fa4e0fc57f6670b63c9bb

FINAL DEVELOPMENT TRAINING HISTORY


,epoch,training_loss,training_patches,batches,batch_size,elapsed_seconds,validation_performed
0,1,0.089836,17895,560,32,344.940912,False
1,2,0.035103,17895,560,32,344.185202,False
2,3,0.023573,17895,560,32,339.212769,False
3,4,0.018708,17895,560,32,338.140556,False
4,5,0.012382,17895,560,32,337.825957,False



STAGE 6 — FINAL COMPLETION AUDIT
PASS: Training resumed from verified checkpoint.
PASS: All five training epochs completed.
PASS: Final Epoch 5 checkpoint saved.
PASS: All five checkpoint SHA-256 hashes verified.
PASS: Cumulative training history verified.
PASS: Optimizer and gradient-scaler states preserved.
PASS: BatchNorm running statistics frozen.
PASS: Final training completion record saved.
PASS: Final backup ZIP verified.
PASS: No validation-based checkpoint selection.
PASS: Calibration, Test and external data untouched.

Stage 6 final training: COMPLETE
Final checkpoint: /content/CIS631_model_checkpoints/efficientnet_b0/final_development/efficientnet_b0_final_epoch5.pt
Final SHA-256: 84ea0871a309f3d68b8958b68b6da4f01adb31d3109fa4e0fc57f6670b63c9bb
Backup: /content/CIS631_EfficientNetB0_Final_Stage6_Backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:

# ============================================================
# Independent Final Model Audit and Metadata Export
# ============================================================

import json
import torch.nn as nn

from torchvision.models import efficientnet_b0


# ------------------------------------------------------------
# 1. Verify the completed training records
# ------------------------------------------------------------

assert FINAL_CHECKPOINT_PATH.is_file()

history = pd.read_csv(FINAL_HISTORY_PATH)

assert history["epoch"].tolist() == [1, 2, 3, 4, 5]
assert len(history) == 5
assert history["validation_performed"].eq(False).all()

completion_path = (
    FINAL_RESULTS_DIR /
    "final_training_completion.json"
)

with open(completion_path, encoding="utf-8") as f:
    completion = json.load(f)

assert completion["training_completed"] is True
assert completion["epochs_completed"] == 5
assert completion["checkpoint_selected"] == "fixed_epoch_5"

# ------------------------------------------------------------
# 2. Independently verify checkpoint SHA-256
# ------------------------------------------------------------

def audit_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


checkpoint_hash = audit_sha256(
    FINAL_CHECKPOINT_PATH
)

assert checkpoint_hash == (
    completion["checkpoint_sha256"]
)

assert checkpoint_hash == (
    "84ea0871a309f3d68b8958b68b6da4f01adb31d3109fa4e0fc57f6670b63c9bb"
)

print("PASS: Final checkpoint SHA-256 verified.")

# ------------------------------------------------------------
# 3. Independently reconstruct the architecture
# ------------------------------------------------------------

# No pretrained download is needed: we load the trained
# weights from our verified final checkpoint.

independent_model = efficientnet_b0(
    weights=None
)

in_features = (
    independent_model.classifier[-1].in_features
)

assert in_features == 1280

independent_model.classifier[-1] = nn.Linear(
    in_features, 1
)

saved_checkpoint = torch.load(
    FINAL_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

assert saved_checkpoint["epoch_completed"] == 5
assert saved_checkpoint["architecture"] == "efficientnet_b0"

independent_model.load_state_dict(
    saved_checkpoint["model_state_dict"],
    strict=True
)

independent_model.eval()

assert sum(
    p.numel()
    for p in independent_model.parameters()
) == 4_008_829

print("PASS: Independent EfficientNet-B0 reconstruction.")
print("PASS: All checkpoint tensors loaded strictly.")

# ------------------------------------------------------------
# 4. Verify forward inference using Development data
# ------------------------------------------------------------

# This is only an operational test, not a performance
# evaluation. No Calibration or Test images are loaded.

sample = final_train_dataset[0]

if isinstance(sample, dict):
    sample_image = sample.get("image")

    if sample_image is None:
        sample_image = sample.get("pixel_values")
else:
    sample_image = sample[0]

assert sample_image.shape == (3, 224, 224)

with torch.inference_mode():

    output_logit = independent_model(
        sample_image.unsqueeze(0).cpu()
    )

    output_probability = torch.sigmoid(
        output_logit
    )

assert output_logit.shape == (1, 1)
assert torch.isfinite(output_logit).all()
assert torch.isfinite(output_probability).all()

print("PASS: Final model produces valid binary logits.")
print("PASS: Sigmoid probabilities are finite.")

# ------------------------------------------------------------
# 5. Verify all five epoch integrity records
# ------------------------------------------------------------

for epoch in range(1, 6):

    checkpoint_path = (
        FINAL_CHECKPOINT_PATH
        if epoch == 5
        else FINAL_CHECKPOINT_DIR /
        f"efficientnet_b0_final_epoch{epoch}_resume.pt"
    )

    integrity_path = (
        FINAL_RESULTS_DIR /
        f"epoch{epoch}_checkpoint_integrity.json"
    )

    assert checkpoint_path.is_file()
    assert integrity_path.is_file()

    with open(integrity_path, encoding="utf-8") as f:
        integrity = json.load(f)

    assert integrity["epoch"] == epoch

    assert audit_sha256(checkpoint_path) == (
        integrity["checkpoint_sha256"]
    )

    assert integrity[
        "batchnorm_running_statistics_unchanged"
    ] is True

print("PASS: All five epoch checkpoint hashes verified.")
print("PASS: BatchNorm audit records verified.")

# ------------------------------------------------------------
# 6. Save independent audit record
# ------------------------------------------------------------

audit_record = {
    "stage": 6,
    "architecture": "efficientnet_b0",
    "training_complete": True,
    "epochs": 5,
    "patients": 49,
    "images": 1133,
    "patches": 17895,
    "selected_checkpoint": "fixed_epoch_5",
    "checkpoint_sha256": checkpoint_hash,
    "independent_model_reconstruction": True,
    "strict_checkpoint_loading": True,
    "forward_pass_verified": True,
    "all_epoch_hashes_verified": True,
    "final_training_loss": float(
        history.iloc[-1]["training_loss"]
    ),
    "interruption_recovery": (
        "Resumed from verified Epoch 1 checkpoint"
    ),
    "persistent_worker_rng_exact_replay": False,
    "calibration_used": False,
    "test_used": False,
    "external_data_used": False
}

audit_path = (
    FINAL_RESULTS_DIR /
    "final_model_independent_audit.json"
)

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(audit_record, f, indent=2)

# ------------------------------------------------------------
# 7. Create GitHub-ready metadata ZIP
# ------------------------------------------------------------

metadata_zip = Path(
    "/content/CIS631_Stage6_GitHub_Metadata.zip"
)

metadata_files = sorted(
    path
    for path in FINAL_RESULTS_DIR.iterdir()
    if path.is_file()
    and path.suffix.lower() in {".csv", ".json"}
)

with zipfile.ZipFile(
    metadata_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in metadata_files:

        zf.write(
            path,
            arcname=str(
                path.relative_to(REPO_ROOT)
            )
        )

with zipfile.ZipFile(metadata_zip) as zf:

    assert zf.testzip() is None

    assert not any(
        name.endswith((".pt", ".pth"))
        for name in zf.namelist()
    )

print("PASS: GitHub metadata ZIP verified.")
print("Metadata files:", len(metadata_files))

# ------------------------------------------------------------
# 8. Final report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STAGE 6 — INDEPENDENT COMPLETION AUDIT")
print("=" * 70)

display(history.round(6))

print("\nFinal checkpoint SHA-256:", checkpoint_hash)

print("\nPASS: Exactly five epochs completed.")
print("PASS: Fixed Epoch 5 checkpoint verified.")
print("PASS: Independent model reconstructed.")
print("PASS: Final model forward inference verified.")
print("PASS: All epoch checkpoint hashes verified.")
print("PASS: Training history and audit saved.")
print("PASS: GitHub export contains metadata only.")
print("PASS: No additional training performed.")
print("PASS: Calibration, Test and external data untouched.")

print("\nStage 6: READY TO CLOSE")
print("Metadata ZIP:", metadata_zip)

del independent_model, saved_checkpoint

files.download(str(metadata_zip))


PASS: Final checkpoint SHA-256 verified.
PASS: Independent EfficientNet-B0 reconstruction.
PASS: All checkpoint tensors loaded strictly.
PASS: Final model produces valid binary logits.
PASS: Sigmoid probabilities are finite.
PASS: All five epoch checkpoint hashes verified.
PASS: BatchNorm audit records verified.
PASS: GitHub metadata ZIP verified.
Metadata files: 10

STAGE 6 — INDEPENDENT COMPLETION AUDIT


,epoch,training_loss,training_patches,batches,batch_size,elapsed_seconds,validation_performed
0,1,0.089836,17895,560,32,344.940912,False
1,2,0.035103,17895,560,32,344.185202,False
2,3,0.023573,17895,560,32,339.212769,False
3,4,0.018708,17895,560,32,338.140556,False
4,5,0.012382,17895,560,32,337.825957,False



Final checkpoint SHA-256: 84ea0871a309f3d68b8958b68b6da4f01adb31d3109fa4e0fc57f6670b63c9bb

PASS: Exactly five epochs completed.
PASS: Fixed Epoch 5 checkpoint verified.
PASS: Independent model reconstructed.
PASS: Final model forward inference verified.
PASS: All epoch checkpoint hashes verified.
PASS: Training history and audit saved.
PASS: GitHub export contains metadata only.
PASS: No additional training performed.
PASS: Calibration, Test and external data untouched.

Stage 6: READY TO CLOSE
Metadata ZIP: /content/CIS631_Stage6_GitHub_Metadata.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>